## Quiet the logs

`scarlets/utils/RedisLogger.py` calls `logging.basicConfig(level=DEBUG)` at
**import time**, which sets the *root* logger to DEBUG for the whole kernel -
so every library that logs (redis-py, httpx, asyncio) starts emitting DEBUG
into your cell output as soon as anything imports `scarlets`.

Run this first. It configures the root logger before `scarlets` gets the
chance, which makes that `basicConfig` call a no-op (`basicConfig` does
nothing when the root logger already has handlers).

If you ever see lines like `Failed to enable maintenance notifications:
unknown subcommand 'MAINT_NOTIFICATIONS'` - those are harmless. redis-py 8.x
probes for a Redis 8 maintenance feature on every new connection, Redis 7.x
replies "unknown subcommand", and the client logs it at DEBUG and carries on
with a perfectly normal connection.

In [ ]:
import logging

# Claim the root logger first. logging.basicConfig() does nothing if the
# root logger already has handlers, so configuring it here makes
# RedisLogger's own basicConfig(level=DEBUG) - which runs when scarlets is
# imported further down - a no-op instead of a takeover.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# Belt and braces: if something already imported scarlets in this kernel
# (a re-run, or an earlier notebook), the call above was itself the no-op,
# so set the level explicitly too.
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

# 10 — Matrix statistics over a heterogeneous fleet

The matrix analog of notebook 08. Where 08 computed scalar statistics over
a single column named `value`, this computes **per-column** statistics over
2-D sensor matrices — and does it across workers that hold **different
columns and different numbers of rows**.

Three things are deliberately different here, and each one is the point:

**No worker is told where its data is.** There is no `CSV_PATH`. Each
worker reads its own `~/.scarlet/config.yaml`, profiles every source it
finds at boot, and writes its own SQL. The paths below are deliberately
irregular — `plant_alpha/turbine_vibration.csv`,
`archive/2026/q3/unit7_sensors.csv` — so nothing can pattern-match a
worker index out of a filename.

**The head never names a column.** It states an objective in plain
language. Which columns exist, and which are readable, is something the
workers know and the head does not.

**The workers hold different columns**, so they must agree on a common
representation before anything can be aggregated. `agree_representation`
is what finds it; `sum` refuses to guess if you skip that step.


## Parameters

In [ ]:
import os, time, json, subprocess, random

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

## Generate each worker's matrix data and its local config

Four workers, four different shapes, four different column sets. The
intersection is never written down anywhere — it has to be discovered.

`HOME` for each worker points into this shared volume, so
`~/.scarlet/config.yaml` resolves to a path this notebook can write while
the worker still reads it from its real default location.


In [ ]:
#Just for simulation. In reality youll have different CSV/other files held locally across siloed plants/equipments that a decentralized.
import numpy as np
import pandas as pd
import yaml

DATA_DIR = os.environ.get("DATA_DIR", "/data")

# Irregular on purpose: differing directory depth, no shared naming
# convention, and source names that carry no worker index. The discovery
# mechanism must rely only on what each config declares.
LAYOUT = {
    "worker1": ("plant_alpha/turbine_vibration.csv",    "turbine_alpha_vibration",  80,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw"]),
    "worker2": ("feeds/grid_telemetry.csv",             "grid_telemetry_feed",     100,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "power_kw"]),
    "worker3": ("archive/2026/q3/unit7_sensors.csv",    "unit7_archive_q3",        120,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw",
                 "ambient_temp_c"]),
    "worker4": ("raw_drop/sensor_export_FINAL.csv",     "bulk_sensor_drop",         90,
                ["vibration_rms", "shaft_speed_rpm", "power_kw"]),
}

# Plausible ranges per sensor, so the numbers mean something.
PARAMS = {
    "vibration_rms":   (2.5,   0.6),
    "bearing_temp_c":  (65.0,  8.0),
    "shaft_speed_rpm": (1800.0, 120.0),
    "torque_nm":       (340.0, 45.0),
    "power_kw":        (64.0,  11.0),
    "ambient_temp_c":  (22.0,  4.0),
}

for i, (node, (relpath, source_name, rows, cols)) in enumerate(LAYOUT.items(), start=1):
    rng = np.random.default_rng(seed=i * 1000)
    start = pd.Timestamp("2026-01-01T00:00:00")
    frame = {"timestamp": [start + pd.Timedelta(minutes=k) for k in range(rows)]}
    for c in cols:
        mean, sd = PARAMS[c]
        frame[c] = np.clip(rng.normal(mean, sd, size=rows), 0.0, None)
    df = pd.DataFrame(frame)

    # Three gaps per file, as any real feed would have. They matter: a row
    # is dropped only when a column being *read* is missing, so the usable
    # row count depends on which columns the fleet agrees on.
    gap_cols = list(rng.permutation(cols))[:3]
    gap_rows = rng.choice(rows, size=3, replace=False)
    for r, c in zip(gap_rows, gap_cols):
        df.at[r, c] = np.nan

    path = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)

    # The worker's own config, at its real ~/.scarlet/config.yaml.
    scarlet_dir = os.path.join(DATA_DIR, node, ".scarlet")
    os.makedirs(scarlet_dir, exist_ok=True)
    actual_gaps = sorted(df.columns[df.isna().any()].tolist())
    with open(os.path.join(scarlet_dir, "config.yaml"), "w") as fh:
        yaml.safe_dump({"sources": [{
            "name": source_name,
            "type": "csv",
            "mode": "local",
            # Derived from the data that was actually written, never
            # hardcoded - a description that disagrees with its file is
            # worse than no description, because the worker's own LLM
            # reads this to reason about what it holds.
            "description": (f"{rows} rows. Columns: {', '.join(cols)}. "
                            f"gaps in columns: {', '.join(actual_gaps)}"),
            "path": path,
        }]}, fh, sort_keys=False)

    print(f"{source_name:26s} {rows:>4} rows x {len(cols)} cols -> {path}")

_sets = [set(v[3]) for v in LAYOUT.values()]
print("\nintersection the fleet will have to discover:",
      sorted(set.intersection(*_sets)))


## Bring the fleet up

In [ ]:
#now we are going to bring the workers up. these can be made fully decentralized using Gustavo!

AGENT_SERVICES = ["agent-head"] + [f"agent-{n}" for n in WORKER_NODES]

def compose(*args):
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result

if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    missing_llm = [k for k in ("LLM_BASE_URL", "LLM_MODEL")
                   if not os.environ.get(k, "").strip()]
    if missing_llm:
        raise RuntimeError(
            f"{', '.join(missing_llm)} not set - the agent head cannot start "
            f"without them.\nSet these in examples/notebooks/.env and restart "
            f"the stack (docker compose up -d), then re-run this notebook."
        )
    
    print("bringing up:", ", ".join(AGENT_SERVICES))
    #compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

## Wait for the fleet to come online

In [ ]:
from scarlets.messaging import Messenger

FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS!r}: "
                f"{sorted(missing)}\nSeen on the bus: {sorted(live_agents())}\n"
                f"Check {where}."
            )
        time.sleep(2)

wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

## Ask the head for per-column statistics

The head is given an objective and nothing else — no source, no path, no
column. It has to run the consensus step first, because `sum` will refuse
to aggregate matrices of differing width.

Mean and variance per column come from the same two-sums composition 08
used, except every quantity is now a vector: `sum(x)` and `sum(x^2)` are
per-column, so mean and variance come out per-column too.


In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# The ordering instruction is load-bearing. `sum` fails deliberately on
# mismatched widths rather than truncating to the narrowest, so the
# agreement has to happen first and its exact column list has to be passed
# through to both sum calls - otherwise each worker reads whatever it
# happens to hold and the shapes will not line up.
answer = await ask_scarlet_agent(
    "Compute per-column mean and variance of the sensor measurements held "
    "across the fleet. "
    "Step 1: call agree_representation to find the columns every worker "
    "can contribute. "
    "Step 2: call sum with transform=identity, passing that exact column "
    "list as `columns`. "
    "Step 3: call sum again with transform=square and the same `columns`. "
    "Step 4: for each column use combine to get mean = s1/n and "
    "variance = s2/n - (s1/n)**2. Remember there is no sqrt() - a standard "
    "deviation is written var**0.5. "
    "Report, per column: the agreed column name, its mean, its variance "
    "and its standard deviation, and state n once."
)
print(answer)


In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# The ordering instruction is load-bearing. `sum` fails deliberately on
# mismatched widths rather than truncating to the narrowest, so the
# agreement has to happen first and its exact column list has to be passed
# through to both sum calls - otherwise each worker reads whatever it
# happens to hold and the shapes will not line up.
answer = await ask_scarlet_agent(
    "Can you tell me if any of the workers have any sources pertaining to tasks? Try to do a query_feature for *_tasks? you can try different combinations."
    "Also tell me if there are any sources relating to turbine, telemetry or vibration information? repeat the same process as before."
)
print(answer)


## Teardown

In [ ]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    #compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")